# Varredura Direta SIGAP

Consulta ao SERPRO/SIGAP **sem passar pela API intermediária** — usa o certificado PFX diretamente.

| | |
|---|---|
| Throughput observado | ~115.000 CPFs em ~6 min com 30 workers |
| Token | Cacheado em disco — sobrevive entre execuções |
| Saída | DataFrame + CSV de impedidos |

In [ ]:
# ============================================================
# CONFIGURAÇÃO — ajuste antes de rodar
# ============================================================

PFX_PATH   = "/home/projects/impedidos/Api_Impedidos/certificates/e-CNPJ_F12.p12"
SENHA_PFX  = ""  # senha do certificado PFX

TOKEN_FILE = "/home/projects/impedidos/Api_Impedidos/certificates/token_gov.json"  # cache compartilhado com a API

AUTH_URL   = "https://auth-sigap-rec.ni.estaleiro.serpro.gov.br/recepcao-autenticacao/token"
SIGAP_URL  = "https://sigap-impedidos.fazenda.gov.br/impedimento/v2/condicao/{cpf}"

WORKERS    = 30   # threads paralelas
TIMEOUT    = 15   # segundos por request

In [ ]:
# ============================================================
# TOKEN — carrega do cache ou obtém novo via PFX
# ============================================================
import json, os, tempfile, time
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from cryptography.hazmat.primitives.serialization import pkcs12, Encoding, PrivateFormat, NoEncryption
from cryptography.hazmat.backends import default_backend


def _extrair_pem(pfx_path: str, senha: str):
    with open(pfx_path, "rb") as f:
        pfx_data = f.read()
    private_key, cert, _ = pkcs12.load_key_and_certificates(
        pfx_data, senha.encode(), backend=default_backend()
    )
    tmp_key  = tempfile.NamedTemporaryFile(delete=False, suffix=".key")
    tmp_cert = tempfile.NamedTemporaryFile(delete=False, suffix=".crt")
    tmp_key.write(private_key.private_bytes(Encoding.PEM, PrivateFormat.TraditionalOpenSSL, NoEncryption()))
    tmp_cert.write(cert.public_bytes(Encoding.PEM))
    tmp_key.close()
    tmp_cert.close()
    return tmp_cert.name, tmp_key.name


def _obter_token_novo():
    """Autentica com o PFX e retorna (token, expires_in_seconds)."""
    cert_path, key_path = _extrair_pem(PFX_PATH, SENHA_PFX)
    try:
        resp = requests.post(AUTH_URL, cert=(cert_path, key_path), verify=True, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        token = data.get("token") or data.get("access_token")
        expires_in = int(data.get("expires_in", 604800))
        return token, expires_in
    finally:
        os.unlink(cert_path)
        os.unlink(key_path)


def carregar_token():
    """Retorna token válido do cache em disco, ou obtém novo."""
    if os.path.exists(TOKEN_FILE):
        try:
            with open(TOKEN_FILE) as f:
                cache = json.load(f)
            expira_em = datetime.fromisoformat(cache["expira_em"])
            if datetime.now() < expira_em - timedelta(seconds=300):
                print(f"[cache] Token válido até {expira_em.strftime('%d/%m/%Y %H:%M')}")
                return cache["token"]
        except Exception:
            pass

    print("Obtendo novo token via PFX...")
    token, expires_in = _obter_token_novo()
    expira_em = datetime.now() + timedelta(seconds=expires_in)
    os.makedirs(os.path.dirname(os.path.abspath(TOKEN_FILE)), exist_ok=True)
    with open(TOKEN_FILE, "w") as f:
        json.dump({"token": token, "expira_em": expira_em.isoformat()}, f)
    print(f"[novo] Token obtido — expira em {expira_em.strftime('%d/%m/%Y %H:%M')}")
    return token


TOKEN = carregar_token()
print(f"Token: {TOKEN[:40]}...")

In [ ]:
# ============================================================
# CONSULTA INDIVIDUAL
# ============================================================

def consultar_cpf(cpf: str, token: str, _retry: bool = True) -> dict:
    cpf = cpf.replace(".", "").replace("-", "").strip()
    headers = {"Authorization": f"Bearer {token}"}
    try:
        resp = requests.get(SIGAP_URL.format(cpf=cpf), headers=headers, timeout=TIMEOUT)
        if resp.status_code == 200:
            d = resp.json()
            return {
                "cpf": cpf,
                "status": d.get("resultado", "INDEFINIDO"),
                "motivos": d.get("motivos", []),
                "data_autoexclusao": d.get("dataSolicitacaoAutoexclusao"),
                "erro": None,
            }
        elif resp.status_code == 401 and _retry:
            global TOKEN
            TOKEN, _ = _obter_token_novo()
            return consultar_cpf(cpf, TOKEN, _retry=False)
        elif resp.status_code == 404:
            return {"cpf": cpf, "status": "NAO_ENCONTRADO", "motivos": [], "data_autoexclusao": None, "erro": None}
        else:
            return {"cpf": cpf, "status": None, "motivos": [], "data_autoexclusao": None, "erro": f"HTTP {resp.status_code}"}
    except Exception as e:
        return {"cpf": cpf, "status": None, "motivos": [], "data_autoexclusao": None, "erro": str(e)}


# Teste rápido com um CPF
resultado = consultar_cpf("00000000000", TOKEN)
print(resultado)

In [ ]:
# ============================================================
# VARREDURA EM LOTE — com progresso
# ============================================================

def varrer_lote(cpfs: list, token: str) -> list:
    total = len(cpfs)
    resultados = []
    erros = 0
    t_inicio = time.time()
    t_ultimo_log = t_inicio

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(consultar_cpf, cpf, token): cpf for cpf in cpfs}
        for i, future in enumerate(as_completed(futures), 1):
            r = future.result()
            resultados.append(r)
            if r["erro"]:
                erros += 1

            agora = time.time()
            if agora - t_ultimo_log >= 5 or i == total:
                elapsed = agora - t_inicio
                cpm = int(i / elapsed * 60) if elapsed > 0 else 0
                pct = 100 * i // total
                print(f"{i:>7}/{total}  {pct:>3}%  |  {cpm:>6} CPFs/min  |  erros: {erros}")
                t_ultimo_log = agora

    elapsed = time.time() - t_inicio
    impedidos = [r for r in resultados if r["status"] == "IMPEDIDO"]
    print(f"\nFinalizado em {elapsed:.1f}s  —  Impedidos: {len(impedidos)}  |  Erros: {erros}")
    return resultados

In [ ]:
# ============================================================
# CARREGAR CPFs — escolha uma opção
# ============================================================
import csv

# --- Opção A: lista manual ---
cpfs = [
    "00000000000",
    "11111111111",
]

# --- Opção B: arquivo CSV com coluna 'cpf' ---
# CSV_PATH = "cpfs.csv"
# with open(CSV_PATH, newline="", encoding="utf-8") as f:
#     reader = csv.DictReader(f)
#     cpfs = [row["cpf"].replace(".","").replace("-","").strip() for row in reader]

# Remove duplicatas e CPFs vazios
cpfs = list({c for c in cpfs if c.strip()})
print(f"{len(cpfs)} CPFs carregados")

In [ ]:
# ============================================================
# EXECUTAR VARREDURA
# ============================================================

resultados = varrer_lote(cpfs, TOKEN)

In [ ]:
# ============================================================
# ANALISAR E EXPORTAR
# ============================================================
import pandas as pd

df = pd.DataFrame(resultados)
df["motivos"] = df["motivos"].apply(lambda x: ", ".join(x) if x else "")

print("=== Distribuição de status ===")
print(df["status"].value_counts())
print()

df_impedidos = df[df["status"] == "IMPEDIDO"].copy()
print(f"=== Impedidos ({len(df_impedidos)}) ===")
display(df_impedidos.head(30))

# Exportar
saida = f"impedidos_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_impedidos.to_csv(saida, index=False)
print(f"\nArquivo exportado: {saida}")